# JetBot Socket Server

This notebook starts a small TCP server on the JetBot so a laptop can send left/right motor commands.

Intended use:
- Run this notebook on the JetBot from JupyterLab.
- Run `scripts/vicon_teleop_viewer.py` on the laptop.
- The laptop handles the Vicon UDP viewer and keyboard teleoperation.

Safety notes:
- The server includes a command timeout watchdog.
- If commands stop arriving, the robot is stopped automatically.
- Closing the client connection should also stop the robot.


In [ ]:
HOST = "0.0.0.0"
PORT = 8765
COMMAND_TIMEOUT_SECONDS = 0.5

print(f"JetBot socket server config: {HOST}:{PORT}, timeout={COMMAND_TIMEOUT_SECONDS}s")


In [ ]:
import json
import socketserver
import threading
import time

from jetbot import Robot


def clamp(value, low=-1.0, high=1.0):
    return max(low, min(high, float(value)))


class JetBotController:
    def __init__(self, timeout_seconds):
        self.robot = Robot()
        self.timeout_seconds = timeout_seconds
        self.lock = threading.Lock()
        self.last_command_time = 0.0
        self.last_client = None
        self.last_command = {"left": 0.0, "right": 0.0}
        self.stop_event = threading.Event()
        self.watchdog_thread = threading.Thread(target=self._watchdog_loop, daemon=True)
        self.watchdog_thread.start()

    def _set_motors(self, left, right):
        self.robot.left_motor.value = clamp(left)
        self.robot.right_motor.value = clamp(right)

    def drive(self, left, right, client_label):
        left = clamp(left)
        right = clamp(right)
        with self.lock:
            self._set_motors(left, right)
            self.last_command_time = time.time()
            self.last_client = client_label
            self.last_command = {"left": left, "right": right}

    def stop(self, reason="stop"):
        with self.lock:
            self._set_motors(0.0, 0.0)
            self.last_command = {"left": 0.0, "right": 0.0}
        print(f"Robot stopped: {reason}")

    def _watchdog_loop(self):
        while not self.stop_event.wait(0.05):
            with self.lock:
                age = time.time() - self.last_command_time if self.last_command_time else None
                moving = self.last_command["left"] != 0.0 or self.last_command["right"] != 0.0
            if age is not None and moving and age > self.timeout_seconds:
                self.stop(f"command timeout after {age:.2f}s")

    def close(self):
        self.stop_event.set()
        self.stop("server shutdown")
        self.watchdog_thread.join(timeout=1.0)


class TeleopTCPServer(socketserver.ThreadingTCPServer):
    allow_reuse_address = True
    daemon_threads = True


class TeleopRequestHandler(socketserver.StreamRequestHandler):
    def handle(self):
        client_label = f"{self.client_address[0]}:{self.client_address[1]}"
        print(f"Client connected: {client_label}")

        try:
            while True:
                raw_line = self.rfile.readline()
                if not raw_line:
                    break

                try:
                    message = json.loads(raw_line.decode("utf-8").strip())
                except json.JSONDecodeError as exc:
                    print(f"Bad JSON from {client_label}: {exc}")
                    continue

                message_type = message.get("type", "")
                controller = self.server.controller

                if message_type == "drive":
                    controller.drive(
                        message.get("left", 0.0),
                        message.get("right", 0.0),
                        client_label,
                    )
                elif message_type == "stop":
                    controller.stop(f"stop request from {client_label}")
                elif message_type == "ping":
                    pass
                else:
                    print(f"Unknown command from {client_label}: {message}")
        except Exception as exc:
            print(f"Client error from {client_label}: {exc}")
        finally:
            self.server.controller.stop(f"client disconnected: {client_label}")
            print(f"Client disconnected: {client_label}")


def stop_server():
    state = globals().get("SERVER_STATE")
    if not state:
        print("No running server.")
        return

    server = state["server"]
    thread = state["thread"]
    controller = state["controller"]

    server.shutdown()
    server.server_close()
    controller.close()
    thread.join(timeout=1.0)
    globals()["SERVER_STATE"] = {}
    print("Server stopped.")


def start_server(host, port, timeout_seconds):
    controller = JetBotController(timeout_seconds=timeout_seconds)
    server = TeleopTCPServer((host, port), TeleopRequestHandler)
    server.controller = controller
    thread = threading.Thread(target=server.serve_forever, daemon=True)
    thread.start()
    globals()["SERVER_STATE"] = {
        "server": server,
        "thread": thread,
        "controller": controller,
    }
    return globals()["SERVER_STATE"]


print("Definitions loaded.")


In [ ]:
stop_server()
SERVER_STATE = start_server(HOST, PORT, COMMAND_TIMEOUT_SECONDS)
print(f"JetBot teleop server running on {HOST}:{PORT}")


Use this on the laptop after the server is running:

```bash
python scripts/vicon_teleop_viewer.py --jetbot-host 10.53.174.144
```

Keep the Matplotlib window focused while driving so the keyboard events are delivered.


In [ ]:
# Run this cell when you want to stop the server.
stop_server()
